# 04. Output Parser

LLM의 출력을 원하는 형식(문자열, JSON, 객체 등)으로 파싱합니다.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(api_key=os.environ["LLM_API_KEY"], base_url=os.getenv("LLM_BASE_URL", "https://monogpt.kr/api/monorouter/v1"), use_responses_api=False, model="gpt-4.1", temperature=0)
if not os.getenv("LLM_API_KEY"):
    raise ValueError("프로젝트 .env에 LLM_API_KEY를 설정하세요.")


## 1. StrOutputParser - 문자열 추출

`AIMessage` 객체에서 `.content` 문자열만 추출합니다.

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("human", "{topic}을 한 문장으로 설명해줘")
])

parser = StrOutputParser()

# prompt -> llm -> parser 순으로 수동 호출
messages = prompt.invoke({"topic": "LangChain"})
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(result))  # str
print(result)

<class 'langchain_core.messages.base.TextAccessor'>
LangChain은 다양한 언어 모델과 데이터 소스를 연결하여 복잡한 자연어 처리 애플리케이션을 쉽게 개발할 수 있도록 도와주는 파이썬 기반 오픈소스 프레임워크입니다.


## 2. JsonOutputParser - JSON 파싱

LLM이 JSON을 반환하도록 유도하고 딕셔너리로 파싱합니다.

In [3]:
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("human", "파이썬 언어에 대한 정보를 name, year, creator 키를 가진 JSON으로 반환해줘. JSON만 반환해.")
])

parser = JsonOutputParser()

messages = prompt.invoke({})
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(result))  # dict
print(result)

<class 'dict'>
{'name': 'Python', 'year': 1991, 'creator': 'Guido van Rossum'}


## 3. PydanticOutputParser - 구조체 파싱

Pydantic 모델을 정의하면 자동으로 포맷 지시사항을 생성하고 파싱합니다.

In [4]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 원하는 데이터의 출력 구조 정의
class MovieReview(BaseModel):
    title: str = Field(description="영화 제목")
    score: int = Field(description="10점 만점 점수")
    summary: str = Field(description="한 줄 감상")

parser = PydanticOutputParser(pydantic_object=MovieReview)

# 파서가 자동으로 포맷 지시사항 생성
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "영화 제목", "title": "Title", "type": "string"}, "score": {"description": "10점 만점 점수", "title": "Score", "type": "integer"}, "summary": {"description": "한 줄 감상", "title": "Summary", "type": "string"}}, "required": ["title", "score", "summary"]}
```


In [5]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 평론가입니다.\n{format_instructions}"),
    ("human",  "{movie} 영화를 리뷰해줘")
])

messages = prompt.invoke({
    "movie": "인터스텔라",
    "format_instructions": parser.get_format_instructions()
})
response = llm.invoke(messages)
result   = parser.invoke(response)

print(type(result))    # <class MovieReview>
print(result.title)    # 인터스텔라
print(result.score)    # 10
print(result.summary)


<class '__main__.MovieReview'>
인터스텔라
9
웅장한 스케일과 깊은 감성, 과학적 상상력이 어우러진 명작 SF 영화.


## 4. CommaSeparatedListOutputParser - 리스트 파싱

In [6]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()

prompt = ChatPromptTemplate.from_messages([
    ("human", "파이썬 웹 프레임워크 Top 5를 쉼표로 구분해서 나열해줘. 설명 없이 이름만.")
])

messages = prompt.invoke({})
response = llm.invoke(messages)
result = parser.invoke(response)

print(type(result))  # list
print(result)

<class 'list'>
['Django', 'Flask', 'FastAPI', 'Tornado', 'Pyramid']
